# Treino Melhorado - MIQR-CC Dataset

Classificação automática de imagens CPRE em 4 classes:
- `Biliary_Leaks`, `Lithiasis`, `Normal`, `Stricture`

**Melhorias face à baseline:**
- EfficientNet-B3 (mais eficiente que B7 para datasets pequenos)
- Fine-tuning em 2 fases (cabeça → backbone completo)
- FocalLoss ponderada por classe
- Early stopping + CosineAnnealingLR
- Avaliação completa: F1-macro, matriz de confusão, AUC-ROC
- Grad-CAM para interpretabilidade

**Baseline de referência:** F1-macro = 0.738 (EfficientNet-B7, Martins et al.)

In [ ]:
# Instala dependências em falta
import subprocess
subprocess.run(['pip', 'install', 'grad-cam', 'livelossplot', 'torchmetrics', 'timm', '--quiet'])

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from torchmetrics.classification import MulticlassF1Score
from livelossplot import PlotLosses
from livelossplot.outputs import MatplotlibPlot

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")

---
## 1. Configuração

In [ ]:
# ============================================================
# CONFIGURAÇÃO — ajusta estes paths ao teu ambiente
# ============================================================
DATASET_DIR = './dataset_processed'  # em vez de './dataset' 
MODEL_DIR    = './models'
MODEL_NAME   = 'efficientnet_b3'    # modelo timm
INPUT_SIZE   = 300                  # EfficientNet-B3 usa 300x300
BATCH_SIZE   = 16
NUM_WORKERS  = 4
SEED         = 42

# Fase 1: treinar só a cabeça
EPOCHS_PHASE1 = 10
LR_PHASE1     = 1e-3

# Fase 2: fine-tuning completo
EPOCHS_PHASE2 = 50
LR_PHASE2     = 1e-5

EARLY_STOP_PATIENCE = 10
# ============================================================

os.makedirs(MODEL_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## 2. Transforms e DataLoaders

Usamos `ImageFolder` para ler directamente das pastas `train/`, `val/`, `test/`.

In [ ]:
from torch.utils.data import WeightedRandomSampler

# Augmentation agressiva para treino
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Datasets
train_ds = ImageFolder(os.path.join(DATASET_DIR, 'train'), transform=train_transforms)
val_ds   = ImageFolder(os.path.join(DATASET_DIR, 'val'),   transform=val_transforms)
test_ds  = ImageFolder(os.path.join(DATASET_DIR, 'test'),  transform=val_transforms)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# --- Oversampling com WeightedRandomSampler ---
train_labels   = [label for _, label in train_ds.samples]
counts         = np.bincount(train_labels)
sample_weights = torch.tensor([1.0 / counts[l] for l in train_labels], dtype=torch.float32)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# train_loader usa sampler (sem shuffle=True, são incompatíveis)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Treino: {len(train_ds)} | Validação: {len(val_ds)} | Teste: {len(test_ds)}")

In [ ]:
# Distribuição das classes e class weights
train_labels = [label for _, label in train_ds.samples]
counts = np.bincount(train_labels)
total  = len(train_labels)

print("Distribuição treino:")
for cls, cnt in zip(CLASS_NAMES, counts):
    print(f"  {cls:20s}: {cnt} ({100*cnt/total:.1f}%)")

# Pesos inversamente proporcionais à frequência
class_weights = torch.tensor(
    [total / (NUM_CLASSES * c) for c in counts],
    dtype=torch.float32
).to(device)

print("\nClass weights:", class_weights.tolist())

# Gráfico de distribuição
plt.figure(figsize=(8, 4))
plt.bar(CLASS_NAMES, counts, color='steelblue')
plt.title('Distribuição das classes — Treino')
plt.ylabel('Nº imagens')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---
## 3. Modelo — EfficientNet-B3

Fine-tuning em **2 fases**:
- Fase 1: backbone congelado, treina apenas a cabeça classificadora
- Fase 2: backbone descongelado, fine-tuning completo com LR muito baixo

In [ ]:
def build_model(num_classes, model_name='efficientnet_b3', freeze_backbone=True):
    """
    Cria EfficientNet-B3 pré-treinado no ImageNet.
    Se freeze_backbone=True, congela tudo exceto a cabeça.
    """
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False

    return model


def unfreeze_model(model):
    """Descongela todos os parâmetros para fine-tuning completo."""
    for param in model.parameters():
        param.requires_grad = True
    return model


# Constrói o modelo (fase 1: backbone congelado)
model = build_model(NUM_CLASSES, MODEL_NAME, freeze_backbone=True)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {trainable:,} / {total_params:,}")

---
## 4. Funções de treino e avaliação

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, f1_metric):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds)
        all_labels.append(labels)

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    epoch_loss = running_loss / len(loader.dataset)
    epoch_f1   = f1_metric(all_preds, all_labels).item()
    return epoch_loss, epoch_f1


@torch.no_grad()
def evaluate(model, loader, loss_fn, f1_metric):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds)
        all_labels.append(labels)

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    epoch_loss = running_loss / len(loader.dataset)
    epoch_f1   = f1_metric(all_preds, all_labels).item()
    return epoch_loss, epoch_f1


def train_phase(model, train_loader, val_loader, optimizer, loss_fn,
                epochs, phase_name, save_path, patience=10):
    """
    Loop de treino com early stopping e guarda o melhor modelo.
    """
    f1_metric = MulticlassF1Score(num_classes=NUM_CLASSES, average='macro').to(device)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    liveloss  = PlotLosses(outputs=[MatplotlibPlot(figpath=f"{save_path}_{phase_name}.png")])

    best_f1 = -1
    no_improve = 0
    history = []

    print(f"\n{'='*50}")
    print(f" {phase_name} — {epochs} épocas")
    print(f"{'='*50}")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, loss_fn, f1_metric)
        val_loss,   val_f1   = evaluate(model, val_loader, loss_fn, f1_metric)
        scheduler.step()

        elapsed = time.time() - t0
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_f1': train_f1, 'val_f1': val_f1})

        liveloss.update({
            'loss': train_loss, 'val_loss': val_loss,
            'F1':   train_f1,   'val_F1':   val_f1
        })
        liveloss.send()

        print(f"Época {epoch:3d}/{epochs} | "
              f"Loss {train_loss:.4f}/{val_loss:.4f} | "
              f"F1 {train_f1:.4f}/{val_f1:.4f} | "
              f"{elapsed:.1f}s")

        # Guarda melhor modelo
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)
            no_improve = 0
            print(f"  ✓ Novo melhor F1-val: {best_f1:.4f} — modelo guardado")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping na época {epoch}")
                break

    print(f"\nMelhor F1-val ({phase_name}): {best_f1:.4f}")
    return pd.DataFrame(history)

---
## 5. Fase 1 — Treino da cabeça classificadora

In [ ]:
SAVE_PATH = os.path.join(MODEL_DIR, f'{MODEL_NAME}_best.pth')

loss_fn   = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1
)

history_p1 = train_phase(
    model, train_loader, val_loader,
    optimizer, loss_fn,
    epochs=EPOCHS_PHASE1,
    phase_name='Fase1_Cabeca',
    save_path=SAVE_PATH,
    patience=EARLY_STOP_PATIENCE
)

---
## 6. Fase 2 — Fine-tuning completo

In [ ]:
# Carrega o melhor modelo da fase 1 e descongela tudo
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model = unfreeze_model(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis (fase 2): {trainable:,}")

optimizer_p2 = torch.optim.Adam(model.parameters(), lr=LR_PHASE2)

history_p2 = train_phase(
    model, train_loader, val_loader,
    optimizer_p2, loss_fn,
    epochs=EPOCHS_PHASE2,
    phase_name='Fase2_FineTuning',
    save_path=SAVE_PATH,
    patience=EARLY_STOP_PATIENCE
)

---
## 7. Avaliação no conjunto de teste

In [ ]:
# Carrega o melhor modelo
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# Métricas
f1_macro = f1_score(all_labels, all_preds, average='macro')
acc      = accuracy_score(all_labels, all_preds)

print(f"\n{'='*40}")
print(f" RESULTADOS NO CONJUNTO DE TESTE")
print(f"{'='*40}")
print(f" Accuracy  : {acc:.4f}")
print(f" F1-macro  : {f1_macro:.4f}  (baseline: 0.738)")
print(f"{'='*40}\n")

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# AUC-ROC (one-vs-rest)
try:
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    print(f"AUC-ROC (macro OvR): {auc:.4f}")
except Exception as e:
    print(f"AUC-ROC não calculado: {e}")

In [ ]:
# Matriz de confusão
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title(f'{MODEL_NAME} — Matriz de Confusão (Teste)\nF1-macro: {f1_macro:.4f}')
plt.tight_layout()
cm_path = os.path.join(MODEL_DIR, f'{MODEL_NAME}_cm.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Matriz de confusão guardada em: {cm_path}")

---
## 8. Grad-CAM — Interpretabilidade (obrigatório)

Gera mapas de calor para visualizar as regiões da imagem que influenciaram a classificação.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Camada alvo para Grad-CAM (última camada convolucional do EfficientNet-B3)
target_layer = model.blocks[-1]

cam = GradCAM(model=model, target_layers=[target_layer])


def generate_gradcam(image_path, true_label=None, n=1):
    """
    Gera e mostra o Grad-CAM para uma imagem.
    image_path: caminho para o ficheiro PNG
    true_label: índice da classe real (opcional)
    """
    # Pré-processa a imagem
    img_pil = Image.open(image_path).convert('RGB').resize((INPUT_SIZE, INPUT_SIZE))
    img_np  = np.array(img_pil).astype(np.float32) / 255.0

    tensor = val_transforms(Image.open(image_path)).unsqueeze(0).to(device)

    # Inferência
    with torch.no_grad():
        output = model(tensor)
        pred_idx  = output.argmax(dim=1).item()
        pred_prob = torch.softmax(output, dim=1)[0, pred_idx].item()

    # Grad-CAM
    targets   = [ClassifierOutputTarget(pred_idx)]
    grayscale = cam(input_tensor=tensor, targets=targets)[0]
    cam_img   = show_cam_on_image(img_np, grayscale, use_rgb=True)

    # Visualização
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(cam_img)
    title = f'Pred: {CLASS_NAMES[pred_idx]} ({pred_prob:.2f})'
    if true_label is not None:
        title += f'\nReal: {CLASS_NAMES[true_label]}'
        color  = 'green' if pred_idx == true_label else 'red'
        axes[1].set_title(title, color=color)
    else:
        axes[1].set_title(title)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


print("Função generate_gradcam() pronta.")
print("Uso: generate_gradcam('path/para/imagem.png', true_label=0)")

In [ ]:
# Gera Grad-CAM para 2 exemplos de cada classe do conjunto de teste
print("Exemplos de Grad-CAM por classe:\n")

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATASET_DIR, 'test', class_name)
    images    = [f for f in os.listdir(class_dir) if f.endswith('.png')]
    samples   = random.sample(images, min(2, len(images)))

    print(f"--- {class_name} ---")
    for img_name in samples:
        generate_gradcam(
            os.path.join(class_dir, img_name),
            true_label=class_idx
        )

---
## 9. Histórico de treino

In [ ]:
# Junta histórico das duas fases
history_p1['fase'] = 'Fase 1 (cabeça)'
history_p2['fase'] = 'Fase 2 (fine-tuning)'
history_p2['epoch'] = history_p2['epoch'] + len(history_p1)
history = pd.concat([history_p1, history_p2], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['epoch'], history['train_loss'], label='Treino')
axes[0].plot(history['epoch'], history['val_loss'],   label='Validação')
axes[0].axvline(len(history_p1), linestyle='--', color='gray', label='Início Fase 2')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].legend()

# F1
axes[1].plot(history['epoch'], history['train_f1'], label='Treino')
axes[1].plot(history['epoch'], history['val_f1'],   label='Validação')
axes[1].axhline(0.738, linestyle=':', color='red', label='Baseline (0.738)')
axes[1].axvline(len(history_p1), linestyle='--', color='gray', label='Início Fase 2')
axes[1].set_title('F1-score macro')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.suptitle(f'{MODEL_NAME} — Curvas de treino', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, f'{MODEL_NAME}_history.png'), dpi=150)
plt.show()